# Create AMR Training Datasets in Google Colab

Notebook này thực hiện các bước:
- đọc `genotype.tsv` và `phenotype.tsv`
- pivot `genotype.tsv` thành ma trận `BioSample x gene/mutation`
- join với `phenotype.tsv` theo `BioSample`
- tách từng kháng sinh thành một bài toán binary classification (`resistant` / `susceptible`)

Kỳ vọng đầu vào:
- `genotype.tsv`
- `phenotype.tsv`


In [ ]:
from pathlib import Path
import gc
import re
import zipfile

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 30)
pd.set_option('display.width', 160)


## 1. Chọn cách nạp file

Có 2 cách phổ biến trên Colab:
1. Upload trực tiếp từ máy
2. Mount Google Drive rồi trỏ tới thư mục chứa file


In [ ]:
# Đổi USE_DRIVE = True nếu bạn muốn đọc file từ Google Drive.
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DATA_DIR = '/content/drive/MyDrive/amr_data'  # Äá»•i thÃ nh thÆ° má»¥c cá»§a báº¡n náº¿u cáº§n.
    DATA_DIR = Path(DRIVE_DATA_DIR)
else:
    from google.colab import files
    uploaded = files.upload()
    DATA_DIR = Path('/content')

GENOTYPE_PATH = DATA_DIR / 'genotype.tsv'  # DÃ¹ng / cá»§a pathlib, khÃ´ng dÃ¹ng //.
PHENOTYPE_PATH = DATA_DIR / 'phenotype.tsv'

assert GENOTYPE_PATH.exists(), f'Khong tim thay {GENOTYPE_PATH}'
assert PHENOTYPE_PATH.exists(), f'Khong tim thay {PHENOTYPE_PATH}'

GENOTYPE_PATH, PHENOTYPE_PATH

In [ ]:
genotype = pd.read_csv(
    GENOTYPE_PATH,
    sep='\t',
    dtype={
        '#BioSample': 'string',
        'Element symbol': 'string',
        'Subtype': 'string',
        'Class': 'string',
        'Subclass': 'string',
        'Method': 'string',
    }
)

phenotype = pd.read_csv(
    PHENOTYPE_PATH,
    sep='\t',
    dtype={
        '#BioSample': 'string',
        'Antibiotic': 'string',
        'Measurement sign': 'string',
        'Resistance phenotype': 'string',
    }
)

print('genotype shape :', genotype.shape)
print('phenotype shape:', phenotype.shape)

display(genotype.head())
display(phenotype.head())

In [ ]:
# Chuẩn hóa tên cột và label.
genotype = genotype.rename(columns={'#BioSample': 'BioSample'})
phenotype = phenotype.rename(columns={'#BioSample': 'BioSample'})

phenotype['Antibiotic'] = phenotype['Antibiotic'].str.strip().str.lower()
phenotype['Resistance phenotype'] = phenotype['Resistance phenotype'].str.strip().str.lower()

# Giữ bài toán binary classification.
phenotype = phenotype[phenotype['Resistance phenotype'].isin(['susceptible', 'resistant'])].copy()
phenotype['label'] = phenotype['Resistance phenotype'].map({'susceptible': 0, 'resistant': 1}).astype('int8')

# Tạo tên feature. Tách POINT ra khỏi AMR thường để dễ diễn giải hơn.
genotype['Subtype'] = genotype['Subtype'].fillna('UNKNOWN').str.upper()
genotype['Element symbol'] = genotype['Element symbol'].astype('string').str.strip()
genotype['feature_name'] = np.where(
    genotype['Subtype'].eq('POINT'),
    'POINT::' + genotype['Element symbol'].astype(str),
    'GENE::' + genotype['Element symbol'].astype(str)
)

print('phenotype labels:', phenotype['Resistance phenotype'].value_counts().to_dict())
print('unique genotype features:', genotype['feature_name'].nunique())

## 2. Pivot genotype thành ma trận `BioSample x gene/mutation`

Mỗi hàng là một `BioSample`.
Mỗi cột là một gene hoặc mutation.
Giá trị là 1 nếu feature xuất hiện, 0 nếu không xuất hiện.


In [ ]:
# Loại bỏ các dòng lặp hoàn toàn cùng BioSample-feature trước khi pivot.
geno_pairs = genotype[['BioSample', 'feature_name']].drop_duplicates()

feature_matrix = (
    pd.crosstab(geno_pairs['BioSample'], geno_pairs['feature_name'])
    .astype('uint8')
    .reset_index()
)

print('feature_matrix shape:', feature_matrix.shape)
display(feature_matrix.head())

del geno_pairs
gc.collect()

In [ ]:
training_long = phenotype.merge(feature_matrix, on='BioSample', how='inner')

print('training_long shape:', training_long.shape)
print('joined BioSamples:', training_long['BioSample'].nunique())
display(training_long[['BioSample', 'Antibiotic', 'Resistance phenotype', 'label']].head())

In [ ]:
summary = (
    training_long.groupby('Antibiotic')
    .agg(
        rows=('BioSample', 'size'),
        biosamples=('BioSample', 'nunique'),
        resistant=('label', 'sum')
    )
    .reset_index()
)
summary['susceptible'] = summary['rows'] - summary['resistant']
summary['resistant_pct'] = (summary['resistant'] / summary['rows'] * 100).round(1)
summary = summary.sort_values(['rows', 'resistant_pct'], ascending=[False, False]).reset_index(drop=True)

display(summary.head(20))

## 3. Tách từng kháng sinh thành một bài toán binary classification

Mỗi kháng sinh sẽ tạo ra một bảng riêng:
- đầu vào `X`: các cột feature `GENE::...` hoặc `POINT::...`
- đầu ra `y`: cột `label` (`0 = susceptible`, `1 = resistant`)


In [ ]:
OUTPUT_DIR = DATA_DIR / 'training_datasets'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLS = [c for c in training_long.columns if c.startswith('GENE::') or c.startswith('POINT::')]
META_COLS = ['BioSample', 'Antibiotic', 'Measurement sign', 'MIC (mg/L)', 'Resistance phenotype', 'label']

print('number of feature columns:', len(FEATURE_COLS))
print('output dir:', OUTPUT_DIR)

In [ ]:
def slugify(text: str) -> str:
    text = text.strip().lower()
    text = re.sub(r'[^a-z0-9]+', '_', text)
    return text.strip('_')

dataset_index = []

for antibiotic, df_ab in training_long.groupby('Antibiotic'):
    df_ab = df_ab.reset_index(drop=True)
    antibiotic_slug = slugify(antibiotic)
    out_path = OUTPUT_DIR / f'{antibiotic_slug}_binary_classification.csv'
    df_ab[META_COLS + FEATURE_COLS].to_csv(out_path, index=False)

    dataset_index.append({
        'antibiotic': antibiotic,
        'file_name': out_path.name,
        'rows': len(df_ab),
        'biosamples': df_ab['BioSample'].nunique(),
        'resistant': int(df_ab['label'].sum()),
        'susceptible': int((1 - df_ab['label']).sum()),
        'resistant_pct': round(float(df_ab['label'].mean() * 100), 1),
    })

dataset_index = pd.DataFrame(dataset_index).sort_values('rows', ascending=False).reset_index(drop=True)
dataset_index.to_csv(OUTPUT_DIR / 'dataset_index.csv', index=False)

display(dataset_index.head(20))
print(f'Saved {len(dataset_index)} antibiotic-specific datasets to {OUTPUT_DIR}')

In [ ]:
# Lưu thêm các file tổng hợp để tái sử dụng.
feature_matrix.to_csv(OUTPUT_DIR / 'feature_matrix_biosample_x_feature.csv', index=False)
training_long[META_COLS + FEATURE_COLS].to_csv(OUTPUT_DIR / 'training_long_all_antibiotics.csv', index=False)

print('Saved master feature matrix and long training dataset.')

In [ ]:
zip_path = DATA_DIR / 'training_datasets.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in OUTPUT_DIR.rglob('*'):
        if path.is_file():
            zf.write(path, arcname=path.relative_to(OUTPUT_DIR.parent))

print('Created zip file:', zip_path)

if not USE_DRIVE:
    from google.colab import files
    files.download(str(zip_path))

## 4. Gợi ý bước tiếp theo

Sau notebook này, bạn có thể:
- chọn 3-5 kháng sinh có class balance tốt để làm mô hình đầu tiên
- train baseline model như Logistic Regression, Random Forest, XGBoost
- phân tích feature importance theo từng kháng sinh
- gom kháng sinh theo nhóm nếu muốn phân tích ở mức class
